**Re-identification review.** How identifiable are stays from the static fields alone: unique stays and small cells on combinations of age, sex, ethnicity, diagnosis, weight, outcome and year, repeat patients, rare categories and values found at one site only. Counts only. The results decide how much coarsening Tier 2 needs.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from datetime import date
from pathlib import Path
import pandas as pd

Paths, and the pooled static table limited to the analysis cohort (stays over 6 hours).

In [ ]:
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
derived_dir = open_picu_dir / "derived"
sys.path.insert(0, str(open_picu_dir / "code"))
import harmonise as H, reid as R


static = pd.read_parquet(derived_dir / "static" / "pooled_static.parquet")
co = H.cohort(static)
static = static.merge(co[["site", "stay_key"]], on=["site", "stay_key"])
print(static.groupby("site").size())

**Uniqueness.** Stays that are alone in their cell, and stays in cells smaller than 10, for each combination. Bands are in `reid.py`.

In [ ]:
uniq = R.uniqueness(static)
display(uniq)

**Repeat patients, rare categories and site-only values.**

In [ ]:
display(R.repeat_patients(static))
display(R.rare_categories(static))
display(R.site_only_values(static))

**Category lists for ethnicity and sex.** Counts per site, with counts under 10 shown as `<10`. Used to design the broad ethnicity groups and to see what the rare sex category is called.

In [ ]:
import static as ST

static_by_site = {s: g for s, g in static.groupby("site")}
eth_table = ST.category_table(static_by_site, "ethnicity")
sex_table = ST.category_table(static_by_site, "sex")

**Candidate Tier 2 coarsening.** No year, patient key replaced by `split_group`, `pim_version` dropped, rare diagnosis groups merged, rare sex values set to missing. Two ethnicity options compared: six broad groups, or the recorded labels with rare ones merged. Counts only.

In [ ]:
coarse_g = R.coarsen(static, ethnicity="group")
coarse_d = R.coarsen(static, ethnicity="detail")
print(R.split_group_sizes(coarse_g))

res_g = R.uniqueness(coarse_g, combos=R.T2_COMBOS).assign(ethnicity="group")
res_d = R.uniqueness(coarse_d, combos=R.T2_COMBOS).assign(ethnicity="detail")
res = pd.concat([res_g, res_d])
display(res[res["site"] == "pooled"][["ethnicity", "combination", "unique_stays", "stays_in_cells_below_10", "share_below_k"]])
display(res[res["site"] != "pooled"][["ethnicity", "combination", "site", "share_below_k"]])

**Coarser bands and local suppression.** Coarser age, weight and length-of-stay bands with the six ethnicity groups, then values suppressed field by field for stays in cells under 10 (stays are all kept). The report shows how many values each field loses.

In [ ]:
coarse2 = R.coarsen(static, ethnicity="group", age_edges=R.AGE_COARSE, weight_edges=R.WEIGHT_COARSE, los_edges=R.LOS_COARSE)
display(R.uniqueness(coarse2, combos=R.T2_COMBOS).query("site == 'pooled'")[["combination", "unique_stays", "stays_in_cells_below_10", "share_below_k"]])

key = ["age_band", "sex", "ethnicity", "diagnosis_group", "died", "weight_band", "los_band"]
supp, rep = R.suppress_to_k(coarse2, key, ["los_band", "weight_band", "ethnicity", "diagnosis_group"])
display(rep)
display(R.uniqueness(supp, combos={"all": key}))

**What is merged.** The ethnicity groups with stays per site (counts under 10 shown as `<10`), and the diagnosis groups with the sites at which each would be merged into `Other (merged)`.

In [ ]:
display(R.group_table(static, "ethnicity"))
display(R.merge_table(static, "diagnosis_group"))

**Figures for the presentation.** Identifiability as fields are added (default against coarse bands), the cost of local suppression, repeat patients, and ethnicity groups by site. Saved to `reports/`, which is git-ignored.

In [ ]:
reports_dir = open_picu_dir / "reports"
variants = {"default bands": R.uniqueness(R.coarsen(static), combos=R.T2_COMBOS),
            "coarse bands": R.uniqueness(coarse2, combos=R.T2_COMBOS)}
R.plot_buildup(variants, path=reports_dir / "reid_buildup.png")
R.plot_suppression(rep, len(coarse2), path=reports_dir / "reid_suppression.png")
R.plot_repeat(static, path=reports_dir / "reid_repeat.png")
R.plot_composition(static, "ethnicity", path=reports_dir / "reid_ethnicity.png")